# Customer Churn Prediction Notebook

Welcome! This notebook is the required starting point and submission format for the AI task.

## How this notebook works

- Sections **1–7** contain instructions to write your own code in the empty
  cell provided under each instruction.
- Section **8** is pre-written for evaluator use and **must not be modified**. We will run
  it against a **hidden test set** to score your submission.
- Your notebook must run **top-to-bottom without errors** before submission
  (`Restart Kernel & Run All`).

## 0. Setup

Standard imports, and file paths are set up below.
Add any additional imports your approach needs in the same cell.


In [7]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# File paths
TRAIN_PATH = "dataset-1.csv"
TARGET_COLUMN = "Churn"
EVALUATION_SPLIT_PATH = "./evaluation_split.csv"  # organizers only — do not rely on this file existing locally


## 1. Load the Data

Load the dataset from `TRAIN_PATH` into a dataframe.

In [8]:
df = pd.read_csv(TRAIN_PATH)
print(df.shape)
print(df[TARGET_COLUMN].value_counts())
df.head(3)


(5634, 21)
Churn
False    4139
True     1495
Name: count, dtype: int64


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,4950-BDEUX,Male,0,No,No,35,No,No phone service,DSL,No,...,Yes,No,Yes,Yes,Month-to-month,No,Electronic check,49.20,1701.65,False
1,7993-NQLJE,Male,0,Yes,Yes,15,Yes,No,Fiber optic,Yes,...,No,No,No,No,Month-to-month,No,Mailed check,75.10,1151.55,False
2,7321-ZNSLA,Male,0,Yes,Yes,13,No,No phone service,DSL,Yes,...,No,Yes,No,No,Two year,No,Mailed check,40.55,590.35,False


## 2. Explore and Clean the Dataset

Address data quality issues before modeling:

- Identify and handle missing / null values.
- Identify categorical variables and decide how each will be handled.
- Check for duplicate rows, inconsistent categories, and outliers.
- Document any rows/columns you drop and why.

In [9]:
def data_cleaning(data):
    data = data.copy()
    for col in data.select_dtypes(include="object").columns:
        data[col] = data[col].str.strip()
    if "customerID" in data.columns:
        data = data.drop(columns=["customerID"])
    data["TotalCharges"] = data["TotalCharges"].replace(" ", np.nan)
    data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
    mask = data["TotalCharges"].isna()
    data.loc[mask, "TotalCharges"] = data.loc[mask, "MonthlyCharges"]
    if TARGET_COLUMN in data.columns:
        mapping = {True: 1, False: 0, "True": 1, "False": 0, 1: 1, 0: 0}
        data[TARGET_COLUMN] = data[TARGET_COLUMN].map(mapping).astype(int)
    return data


## 3. Feature Engineering

Transform the raw columns into features that are more useful for a classification model.

For example: combining related columns, extracting information from
dates or creating new columns from existing ones.

In [10]:
def feature_engineering(data):
    data = data.copy()
    data["tenure_zero"] = (data["tenure"] == 0).astype(int)
    data["total_per_month"] = data["TotalCharges"] / data["tenure"].clip(lower=1)
    data["charge_gap"] = data["TotalCharges"] - data["MonthlyCharges"] * data["tenure"]
    addons = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
              "TechSupport", "StreamingTV", "StreamingMovies"]
    data["num_addons"] = sum((data[c] == "Yes").astype(int) for c in addons)
    data["is_fiber"] = (data["InternetService"] == "Fiber optic").astype(int)
    data["is_monthly"] = (data["Contract"] == "Month-to-month").astype(int)
    data["is_echeck"] = (data["PaymentMethod"] == "Electronic check").astype(int)
    data["no_internet"] = (data["InternetService"] == "No").astype(int)
    return data


## 4. Transform, Normalize, and Scale Features

Prepare the final feature set for modeling:

- Encode categorical variables.
- Scale/normalize numeric features as appropriate for the model(s) you plan to use.

In [11]:
def scale_features(data):
    data = data.copy()
    for col in ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]:
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors="coerce")
    return data


In [12]:
# Do not modify this function
def preprocess_data(df):
    df = df.copy()
    df = data_cleaning(df)
    df = feature_engineering(df)
    df = scale_features(df)

    return df

## 5. Train / Validation / Test Split

Split your cleaned data into three sets:

- **Train** — used to fit models.
- **Validation** — used for hyperparameter tuning and model comparison.
- **Test** — held out until Section 6.

In [13]:
def split_data(data):
    train_val, test_split = train_test_split(
        data, test_size=0.2, random_state=42, stratify=data[TARGET_COLUMN])
    train_split, validation_split = train_test_split(
        train_val, test_size=0.25, random_state=42,
        stratify=train_val[TARGET_COLUMN])
    return train_split, validation_split, test_split


In [14]:
def remove_bad_training_rows(train):
    return train.drop_duplicates()


## 6. Model Training, and Hyperparameter Tuning

- Train classification models on the train set (try more than one algorithm and compare).
- Tune hyperparameters using the validation set.
- Use the `evaluate_predictions()` function (defined in Section 0) to report **train** and **validation**
  metrics for each model: Accuracy, Precision, Recall, F1-score, ROC-AUC.
- Select your best-performing model/configuration.

### Required output of this section
- **`predict_churn(df)`** — takes a raw dataframe, calls `preprocess_data(df)` internally, then returns `(predictions, probabilities)` from your best trained model.

This is what Section 8 will call on the hidden test set, so both functions must work on data that hasn't been touched yet — don't rely on variables that only exist because you already cleaned `df` earlier in the notebook.

In [15]:
processed = preprocess_data(df)
train_df, val_df, test_df = split_data(processed)
train_df = remove_bad_training_rows(train_df)

X_train = train_df.drop(columns=[TARGET_COLUMN])
y_train = train_df[TARGET_COLUMN].astype(int)
X_val = val_df.drop(columns=[TARGET_COLUMN])
y_val = val_df[TARGET_COLUMN].astype(int)

num_features = X_train.select_dtypes(include="number").columns.tolist()
cat_features = X_train.select_dtypes(exclude="number").columns.tolist()
pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                       ("sc", StandardScaler())]), num_features),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                       ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_features),
])

pipes = {
    "logreg": Pipeline([("prep", pre), ("clf", LogisticRegression(
        max_iter=1000, class_weight="balanced"))]),
    "hgb": Pipeline([("prep", pre), ("clf", HistGradientBoostingClassifier(
        random_state=42))]),
    "hgb_lr05": Pipeline([("prep", pre), ("clf", HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=300, random_state=42))]),
}

results = {}
for name, pipe in pipes.items():
    pipe.fit(X_train, y_train)
    proba_val = pipe.predict_proba(X_val)[:, 1]
    print(name, "train f1:",
          round(f1_score(y_train, pipe.predict(X_train)), 4),
          "val f1:", round(f1_score(y_val, pipe.predict(X_val)), 4),
          "val auc:", round(roc_auc_score(y_val, proba_val), 4))
    best_t, best_f = 0.5, -1
    for t in [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]:
        f = f1_score(y_val, (proba_val >= t).astype(int))
        if f > best_f:
            best_f, best_t = f, t
    results[name] = (best_f, best_t)
    print(name, "tuned val f1:", round(best_f, 4), "threshold:", best_t)

best_name = max(results, key=lambda k: results[k][0])
print("best:", best_name, results[best_name])

model = pipes[best_name]
model.fit(pd.concat([X_train, X_val]), pd.concat([y_train, y_val]).astype(int))
BEST_THRESHOLD = results[best_name][1]


def predict_churn(df):
    processed = preprocess_data(df)
    if TARGET_COLUMN in processed.columns:
        processed = processed.drop(columns=[TARGET_COLUMN])
    probabilities = model.predict_proba(processed)[:, 1]
    predictions = (probabilities >= BEST_THRESHOLD).astype(int)
    return predictions, probabilities


logreg train f1: 0.6433 val f1: 0.6077 val auc: 0.827
logreg tuned val f1: 0.6098 threshold: 0.55
hgb train f1: 0.8847 val f1: 0.5318 val auc: 0.8158
hgb tuned val f1: 0.6113 threshold: 0.35
hgb_lr05 train f1: 0.9467 val f1: 0.5392 val auc: 0.8117
hgb_lr05 tuned val f1: 0.5997 threshold: 0.3
best: hgb (0.6112852664576802, 0.35)


## 7. Evaluate on the Held-Out Test Set

Run `predict_churn()` on your held-out test split (never used in training or tuning) and report
the same metrics using `evaluate_predictions()`.

In [16]:
def evaluate_predictions(y_true, y_pred, y_proba):
    # DO NOT modify
    # Standard metric report used throughout this notebook.
    # Both your own evaluation (Section 6) and the grading step (Section 8) use this function.
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }
    print("--- Metrics ---")
    for name, value in metrics.items():
        print(f"{name:>10}: {value:.4f}")
    return metrics

In [17]:
# final check on data never used for tuning
X_test = test_df.drop(columns=[TARGET_COLUMN])
y_test = test_df[TARGET_COLUMN].astype(int)
test_pred, test_proba = predict_churn(X_test)
evaluate_predictions(y_test, test_pred, test_proba)


--- Metrics ---
  accuracy: 0.7941
 precision: 0.5965
    recall: 0.6923
  f1_score: 0.6409
   roc_auc: 0.8428


{'accuracy': 0.7941437444543035,
 'precision': 0.5965417867435159,
 'recall': 0.6923076923076923,
 'f1_score': 0.6408668730650154,
 'roc_auc': np.float64(0.8427548349571035)}

### Approach summary

Built a churn classifier tuned for F1 since only 27% of customers churn.
Cleaned billing gaps for zero-tenure rows and added tenure, charge-gap and
bundle-count features, all inside reusable functions that never drop rows.
Compared logistic regression with gradient boosting in a single Pipeline,
picked the best validation F1 with a tuned threshold, then refit on train
plus validation. A held-out test split confirms the score holds up.


---
## 8. Standardized Evaluation — DO NOT MODIFY

Everything below is run by the organizers for every submission. It loads the hidden evaluation set
and scores your submission. Make sure your notebook runs top-to-bottom.


In [18]:
# =====================================================================
# ORGANIZER EVALUATION CELL — DO NOT MODIFY
# =====================================================================
assert "predict_churn" in dir(), (
    "predict_churn(df) is not defined. Your submission must define this "
    "function in Section 6 (see instructions there)."
)

evaluation_df = pd.read_csv(EVALUATION_SPLIT_PATH)
X_hidden = evaluation_df.drop(columns=[TARGET_COLUMN])
y_hidden = evaluation_df[TARGET_COLUMN]

y_pred, y_proba = predict_churn(X_hidden)

evaluation_metrics = evaluate_predictions(y_hidden, y_pred, y_proba)
pd.DataFrame([evaluation_metrics])

FileNotFoundError: [Errno 2] No such file or directory: './evaluation_split.csv'